# 3.7 and proposed 3.10 — Framing and regional sharpness
The same supplied subject region drives both measurements. Synthetic examples use known boxes; real photographs use manually supplied boxes. **No automatic subject detector is included.** Missing or unreliable regions are flagged, never silently treated as image-center subjects.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from brandlens.analysis.framing import measure_framing
from brandlens.analysis.subject_region import SubjectRegion
from matplotlib.patches import Rectangle
from PIL import Image, ImageFilter

from brandlens.analysis.sharpness import measure_sharpness

## 1. Known framing examples
Bounding boxes are normalized `[x, y, width, height]` in the standardized image. Offsets use Euclidean distances on normalized x/y axes; they are not distances in pixels. The closest thirds intersection is used. Centering and thirds placement are reported without claiming one is universally better.

In [ ]:
boxes = [
    ("Centered", SubjectRegion(.35, .35, .3, .3, source="synthetic")),
    ("Thirds intersection", SubjectRegion(1/3-.1, 1/3-.1, .2, .2, source="synthetic")),
    ("Touches left edge", SubjectRegion(0, .35, .2, .3, source="synthetic")),
    ("Unavailable", None),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, region) in zip(axes.flat, boxes):
    image = Image.new("RGB", (320, 240), (210, 215, 225))
    if region is not None:
        image.paste((40, 100, 160), region.pixel_bounds(image))
    ax.imshow(image, extent=(0, 1, 1, 0))
    for value in (1/3, 2/3):
        ax.axvline(value, color="white", linestyle="--")
        ax.axhline(value, color="white", linestyle="--")
    if region:
        ax.add_patch(Rectangle((region.x, region.y), region.width, region.height,
                               fill=False, edgecolor="orange", linewidth=2))
    result = measure_framing(image, region)
    print(name, json.dumps(result, indent=2))
    ax.set_title(name + " | " + result["status"])
    image.close()
plt.tight_layout()
plt.show()

## 2. Controlled subject/background sharpness
Create texture, then selectively blur the subject or background. The orange box is a supplied rectangular subject proxy. Responses at the region boundary and image border are excluded so the splice itself does not create measured sharpness.

The fixed method uses Pillow grayscale, OpenCV `Laplacian(ksize=1, CV_64F)`, and population variance. No resizing or smoothing is applied by the measurement function. Higher variance may also reflect texture, noise or contrast.

In [ ]:
yy, xx = np.indices((192, 192))
gray = np.where((xx // 4 + yy // 4) % 2, 220, 30).astype(np.uint8)
sharp = Image.fromarray(np.repeat(gray[:, :, None], 3, axis=2))
blurred = sharp.filter(ImageFilter.GaussianBlur(2))
region = SubjectRegion(.25, .25, .5, .5, source="synthetic")
bounds = region.pixel_bounds(sharp)
subject_sharp = blurred.copy()
with sharp.crop(bounds) as patch:
    subject_sharp.paste(patch, bounds)
subject_blurred = sharp.copy()
with blurred.crop(bounds) as patch:
    subject_blurred.paste(patch, bounds)
examples = [("Sharp everywhere", sharp), ("Sharp subject / blurred background", subject_sharp),
            ("Blurred subject / sharp background", subject_blurred), ("Blurred everywhere", blurred)]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, image) in zip(axes.flat, examples):
    result = measure_sharpness(image, region)
    values = result["measurements"]
    ax.imshow(image)
    ax.add_patch(Rectangle((bounds[0], bounds[1]), bounds[2]-bounds[0], bounds[3]-bounds[1],
                           fill=False, edgecolor="orange", linewidth=2))
    ratio = values["sharpness_ratio"]
    ratio_text = "unavailable" if ratio is None else f"{ratio:.3f}"
    ax.set_title(f"{name}\nSubject={values['subject_sharpness']:.2f} | Background={values['background_sharpness']:.2f}\nRatio={ratio_text}")
    ax.axis("off")
    print(name, json.dumps(result, indent=2))
plt.tight_layout()
plt.show()
assert measure_sharpness(subject_sharp, region)["measurements"]["sharpness_ratio"] > 1
assert measure_sharpness(subject_blurred, region)["measurements"]["sharpness_ratio"] < 1

## 3. Unsupported and partial cases
A missing or unreliable box is not a failed composition score. Full-frame boxes have no background. Constant backgrounds have zero variance and an undefined ratio. Keep any available measurements, represent unavailable values as `None`, and route the criterion for review.

In [ ]:
with Image.new("RGB", (64, 64), "gray") as uniform:
    cases = [
        ("Missing region", sharp, None),
        ("Unreliable region", sharp, SubjectRegion(.25, .25, .5, .5, reliable=False)),
        ("No background", sharp, SubjectRegion(0, 0, 1, 1)),
        ("Zero background variance", uniform, region),
        ("Tiny subject", sharp, SubjectRegion(.5, .5, .001, .001)),
    ]
    for name, image, box in cases:
        result = measure_sharpness(image, box)
        print(name, json.dumps(result, indent=2, allow_nan=False))
        assert result["status"] == "unavailable_needs_review"

## 4. Optional real image with a manually identified subject
Set the path and a box measured on the **standardized image**, after EXIF rotation. Coordinates from the unrotated original cannot be reused without transforming them. A box is not a subject segmentation mask and can contain background.

No automatic detector or reliability threshold is invented here. If the intended subject is ambiguous, pass `None` or `reliable=False`.

In [ ]:
IMAGE_PATH = None  # e.g. Path("data/photo.jpg")
SUBJECT_XYWH = None  # e.g. (0.25, 0.15, 0.35, 0.65), only after checking the image

if IMAGE_PATH is not None:
    from brandlens.preprocessing.image_io import load_image
    from brandlens.preprocessing.standardization import (
        StandardizationSettings,
        standardize_image,
    )

    with load_image(IMAGE_PATH) as original:
        image = standardize_image(original, StandardizationSettings(max_size=None))
    try:
        box = SubjectRegion(*SUBJECT_XYWH, source="manual") if SUBJECT_XYWH is not None else None
        results = {"framing_subject_position": measure_framing(image, box),
                   "sharpness_subject_emphasis": measure_sharpness(image, box)}
        print(json.dumps(results, indent=2, allow_nan=False))
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(image)
        if box:
            left, top, right, bottom = box.pixel_bounds(image)
            ax.add_patch(Rectangle((left, top), right-left, bottom-top, fill=False, edgecolor="orange"))
        ax.axis("off")
        plt.show()
    finally:
        image.close()
else:
    print("Set IMAGE_PATH and optionally SUBJECT_XYWH to run this section.")

## Completion boundary
These modules implement measurements and unavailable-case handling. Automatic subject localization and mapping these local result envelopes into the API remain integration work. No Jira status is changed by this demo.

Native resolution is retained and recorded. Choose and keep a consistent preprocessing resolution before calibrating sharpness thresholds. Sharpness ratio alone does not establish visual dominance; intentional motion blur must not automatically fail.